# Combining augmentation scores for 10 example Alus

`run_supremo_example.sh` scores each Alu deletion under six sequence augmentations:
shifts of 0, +1 and -1 bp, each forward and reverse complemented. SuPreMo writes one
column per augmentation, so disruption for a single Alu is six MSE values and six
correlations rather than one of each.

This notebook collapses those six into a mean and standard deviation per Alu, and plots
the result. The spread across augmentations is the useful part: it shows how stable a
prediction is to shifting and flipping the input window.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXAMPLE_DIR = Path.cwd()
INPUT_PATH = EXAMPLE_DIR / 'sample_input_10alus.txt'
# swap to EXAMPLE_DIR / 'output' / 'sample_10alus_scores' after running the script
SCORES_PATH = EXAMPLE_DIR / 'expected_output' / 'sample_10alus_scores'


## Load the input table and the scores


In [ ]:
alus = pd.read_csv(INPUT_PATH, sep='\t')
scores = pd.read_csv(SCORES_PATH, sep='\t')

print(f'{len(alus)} Alus, {len(scores)} scored rows')
scores.head(3)


## Collapse the six augmentations

`var_index` in the scores lines up with `orig_idx` in the input. The augmentation columns
are picked up by prefix so this works unchanged if the shift set changes.


In [ ]:
mse_cols = [c for c in scores.columns if c.startswith('mse_')]
corr_cols = [c for c in scores.columns if c.startswith('corr_')]
print(f'{len(mse_cols)} MSE columns: {mse_cols}')
print(f'{len(corr_cols)} correlation columns: {corr_cols}')

summary = pd.DataFrame({
    'orig_idx': scores['var_index'],
    'mse_mean': scores[mse_cols].mean(axis=1),
    'mse_std': scores[mse_cols].std(axis=1),
    'corr_mean': scores[corr_cols].mean(axis=1),
    'corr_std': scores[corr_cols].std(axis=1),
})

summary = alus[['orig_idx', 'CHROM', 'POS', 'END', 'SVLEN', 'strand', 'repName']].merge(
    summary, on='orig_idx')
summary['family'] = summary['repName'].str[:4]
summary['label'] = summary['repName'] + '  ' + summary['CHROM'] + ':' + summary['POS'].astype(str)
summary = summary.sort_values('mse_mean', ascending=False).reset_index(drop=True)
summary[['label', 'SVLEN', 'mse_mean', 'mse_std', 'corr_mean', 'corr_std']]


## Plot

Two panels rather than one with two y-axes: MSE and correlation are on different scales,
and overlaying them would invite false comparison. Points are coloured by Alu family
(AluY / AluS / AluJ); the exact subfamily is written on the axis, so identity never
depends on colour alone. Bars are the standard deviation across the six augmentations.


In [ ]:
# categorical palette: three validated hues, assigned in fixed order
FAMILY_COLORS = {'AluY': '#2a78d6', 'AluS': '#eb6834', 'AluJ': '#1baf7a'}
INK, INK_MUTED, GRID = '#0b0b0b', '#52514e', '#d8d7d2'

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), sharey=True)
y = np.arange(len(summary))[::-1]  # highest disruption at the top
colors = summary['family'].map(FAMILY_COLORS).fillna('#8a8a8a')

panels = [
    (axes[0], 'mse_mean', 'mse_std', 'Disruption (MSE)'),
    (axes[1], 'corr_mean', 'corr_std', 'Correlation with reference'),
]
for ax, mean_col, std_col, xlabel in panels:
    ax.errorbar(summary[mean_col], y, xerr=summary[std_col], fmt='none',
                ecolor=colors, elinewidth=2, capsize=0, zorder=2)
    ax.scatter(summary[mean_col], y, s=64, c=colors, zorder=3,
               edgecolors='white', linewidths=1.5)
    ax.set_xlabel(xlabel, fontsize=9, color=INK)
    ax.xaxis.grid(True, color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8, colors=INK_MUTED)
    ax.tick_params(axis='y', length=0)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color(GRID)

axes[0].set_yticks(y)
axes[0].set_yticklabels(summary['label'], fontsize=8, color=INK)

handles = [plt.Line2D([], [], marker='o', linestyle='', markersize=8,
                      markerfacecolor=c, markeredgecolor='white', label=f)
           for f, c in FAMILY_COLORS.items() if (summary['family'] == f).any()]
axes[1].legend(handles=handles, frameon=False, fontsize=8, labelcolor=INK_MUTED,
               loc='lower left', title=None)

fig.suptitle('Predicted disruption for 10 example Alus, mean \u00b1 SD over 6 augmentations',
             fontsize=10, color=INK, y=1.0)
fig.tight_layout()
fig.savefig(EXAMPLE_DIR / 'sample_10alus_scores.pdf', bbox_inches='tight')
plt.show()


## Predicted contact maps

`--get_maps` writes the predicted maps alongside the scores. Each entry holds the upper
triangle of the reference and the alternate (Alu-deleted) map, so the square matrix has to
be rebuilt before plotting. The shipped sample holds the unshifted, forward-strand
prediction for each of the 10 Alus; a full run writes one entry per augmentation.


In [ ]:
MAPS_PATH = EXAMPLE_DIR / 'expected_output' / 'sample_10alus_maps.npz'
maps = np.load(MAPS_PATH)
MATRIX_LEN, NUM_DIAGS = int(maps['matrix_len']), int(maps['num_diags'])
print(f"{maps['ref'].shape[0]} maps, {MATRIX_LEN}x{MATRIX_LEN} with {NUM_DIAGS} masked diagonals")


def from_upper_triu(vec, matrix_len=MATRIX_LEN, num_diags=NUM_DIAGS):
    """Rebuild the full symmetric matrix from Akita's upper-triangle vector."""
    z = np.zeros((matrix_len, matrix_len), dtype=float)
    z[np.triu_indices(matrix_len, num_diags)] = vec
    z = z + z.T
    idx = np.arange(matrix_len)
    z[np.abs(idx[:, None] - idx[None, :]) < num_diags] = np.nan
    return z


map_by_idx = {int(v): i for i, v in enumerate(maps['var_index'])}


### Reference, alternate and difference for the most disrupted Alu

Akita predicts log(observed/expected) contact, so the maps are already centred on zero and
are drawn with a diverging scale. The difference map is what the MSE summarises.


In [ ]:
top = summary.iloc[0]
i = map_by_idx[int(top['orig_idx'])]
ref_map = from_upper_triu(maps['ref'][i])
alt_map = from_upper_triu(maps['alt'][i])
diff_map = alt_map - ref_map

lim = np.nanmax(np.abs(ref_map))
dlim = np.nanmax(np.abs(diff_map))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, m, title, v in [
    (axes[0], ref_map, 'Reference', lim),
    (axes[1], alt_map, f"Alu deleted ({top['repName']})", lim),
    (axes[2], diff_map, 'Difference', dlim),
]:
    im = ax.imshow(m, cmap='RdBu_r', vmin=-v, vmax=v, interpolation='none')
    ax.set_title(title, fontsize=9, color=INK)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=7,
                                                                    colors=INK_MUTED)

fig.suptitle(f"{top['label']}  |  MSE {top['mse_mean']:.4g}", fontsize=10, color=INK)
fig.tight_layout()
fig.savefig(EXAMPLE_DIR / 'sample_10alus_topmap.pdf', bbox_inches='tight')
plt.show()


### Difference maps for all 10

Ordered by mean MSE, highest first, on a shared colour scale so panels are comparable.


In [ ]:
order = summary['orig_idx'].tolist()
diffs = [from_upper_triu(maps['alt'][map_by_idx[o]]) - from_upper_triu(maps['ref'][map_by_idx[o]])
         for o in order]
shared = np.nanmax([np.nanmax(np.abs(d)) for d in diffs])

fig, axes = plt.subplots(2, 5, figsize=(11, 4.8))
for ax, d, (_, row) in zip(axes.ravel(), diffs, summary.iterrows()):
    im = ax.imshow(d, cmap='RdBu_r', vmin=-shared, vmax=shared, interpolation='none')
    ax.set_title(f"{row['repName']}\nMSE {row['mse_mean']:.3g}", fontsize=8, color=INK)
    ax.set_xticks([]); ax.set_yticks([])

cbar = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
cbar.ax.tick_params(labelsize=7, colors=INK_MUTED)
cbar.set_label('Alu deleted \u2212 reference', fontsize=8, color=INK_MUTED)
fig.suptitle('Predicted change in contact on deleting each Alu', fontsize=10, color=INK)
fig.savefig(EXAMPLE_DIR / 'sample_10alus_diffmaps.pdf', bbox_inches='tight')
plt.show()


## How variable is a score across augmentations?

A quick sense of whether the spread is small relative to the differences between Alus.


In [ ]:
cv = (summary['mse_std'] / summary['mse_mean'])
print(f'MSE: coefficient of variation across augmentations, median {cv.median():.1%}')
print(f'MSE: spread across the 10 Alus, {summary["mse_mean"].min():.4g} to {summary["mse_mean"].max():.4g}')
print(f'Correlation: SD across augmentations, median {summary["corr_std"].median():.4g}')
